# Klassifikation: Welche Räder brauchen bald Wartung?

**CRISP-DM-Block, Modeling-Phase (Block 05, Folie „Klassifikation schätzt eine
Kategorie")**

## Worum es geht

Im ersten Notebook haben wir eine **Zahl** vorhergesagt (Fahrtdauer). Jetzt sagen wir
eine **Kategorie** vorher: Wird ein bestimmtes Rad in absehbarer Zeit eine
Schadensmeldung auslösen — ja oder nein?

## Was Klassifikation von Regression unterscheidet

*"Bei der Klassifikation geht es darum, Elemente anhand ihrer Merkmale automatisiert in
Klassen einzuteilen. Die Klassen sind vorgegeben, die Klassenzugehörigkeit eines Elements
ist nicht bekannt."* (Provost, F., Fawcett, T. (2015): Data Science für Unternehmen,
S. 45 f.) Die Klassen hier sind denkbar einfach: **"hat schon eine Meldung"** oder
**"noch keine Meldung"**.

## Warum das ein echtes Label ist, kein künstliches

Auf der VeloCity-Modeling-Folie steht dazu ein wichtiger Punkt: Das Label kommt aus der
**echten Geschäftslogik** (`schadensmeldung.csv`), nicht aus einer erfundenen
Zielspalte. Ein Rad hat entweder schon einmal eine Schadensmeldung ausgelöst oder nicht
— das ist ein Fakt aus dem Betrieb, kein Konstrukt für die Übung.

## Woher die Daten kommen

`fahrrad.csv`, `ausleihe.csv` und `schadensmeldung.csv` sind wie im ersten Notebook
**erfunden**, aber mit einem eingebauten, überprüfbaren Zusammenhang zwischen Nutzung und
Meldehäufigkeit (siehe `analytics/README.md`: die Korrelation zwischen kumulierten
Kilometern und Meldungen je Rad liegt bei r ≈ 0,52 — spürbar, aber nicht perfekt, genau
wie in der echten Welt).

## Lernziele

1. Merkmale auf einer *anderen* Ebene bilden als in Notebook 1: nicht pro Fahrt, sondern
   pro Rad (Aggregation mit `.groupby()`)
2. Ein Klassifikationslabel aus einer separaten Tabelle ableiten
3. Einen Entscheidungsbaum trainieren und visualisieren
4. Eine Confusion-Matrix lesen und die Kosten von Fehlalarm und verpasstem Alarm
   gegeneinander abwägen — nicht nur eine einzelne Genauigkeits-Kennzahl anschauen

## Schritt 1 — Bibliotheken importieren

Neu gegenüber Notebook 1: `DecisionTreeClassifier` statt `LinearRegression`, `plot_tree`
zur Visualisierung, und `confusion_matrix` statt `mean_absolute_error` — jedes Verfahren
hat sein eigenes Werkzeug, wie es auf der Modeling-Überblicksfolie in Block 05 heißt.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

pd.set_option("display.max_columns", 20)
print("Bibliotheken geladen.")

## Schritt 2 — Drei Tabellen laden

Diesmal brauchen wir drei Quellen:
- **`fahrrad.csv`**: ein Rad pro Zeile — Typ, Anschaffungsdatum
- **`ausleihe.csv`**: eine Fahrt pro Zeile — daraus bauen wir die Nutzungsmerkmale je Rad
- **`schadensmeldung.csv`**: eine Meldung pro Zeile — daraus bauen wir das Label

**TODO:** Laden Sie alle drei Tabellen. Parsen Sie Datumsspalten wo sinnvoll
(`angeschafft_am` bei `fahrrad`, `startzeit` bei `ausleihe`, `gemeldet_am` bei
`schadensmeldung`).

In [ ]:
fahrrad = pd.read_csv("https://raw.githubusercontent.com/swrobuts/velocity-fallstudie/main/analytics/fahrrad.csv", parse_dates=["angeschafft_am"])
ausleihe = pd.read_csv("https://raw.githubusercontent.com/swrobuts/velocity-fallstudie/main/analytics/ausleihe.csv", parse_dates=["startzeit"])
schadensmeldung = pd.read_csv("https://raw.githubusercontent.com/swrobuts/velocity-fallstudie/main/analytics/schadensmeldung.csv", parse_dates=["gemeldet_am"])

print("Räder:", len(fahrrad))
print("Fahrten:", len(ausleihe))
print("Schadensmeldungen:", len(schadensmeldung))
assert len(fahrrad) == 220, "Es sollten 220 Raeder sein"
fahrrad.head()

## Schritt 3 — Nutzungsmerkmale je Rad bilden

In Notebook 1 hatte jede Zeile eine Fahrt. Jetzt brauchen wir eine Zeile **pro Rad** —
mit zusammengefassten Kennzahlen über alle seine Fahrten: Wie oft wurde es gefahren?
Wie viele Kilometer insgesamt (soweit gemessen)?

Das ist eine **Aggregation**: aus vielen Fahrtzeilen wird eine Zusammenfassungszeile je
Rad. `pandas` macht das mit `.groupby("fahrrad_id")` gefolgt von einer
Aggregationsfunktion.

**TODO:**
1. Gruppieren Sie `ausleihe` nach `fahrrad_id`.
2. Bilden Sie je Rad: die Anzahl Fahrten (`.size()` oder `.count()` einer beliebigen
   Spalte) und die Summe der gemessenen Kilometer (`distanz_km`, mit `.sum()` —
   `pandas` ignoriert dabei fehlende Werte automatisch, das ist genau richtig: wir
   wollen die Summe der *gemessenen* Kilometer, nicht so tun, als wären die
   ungemessenen 0 km gefahren).

In [ ]:
nutzung = ausleihe.groupby("fahrrad_id").agg(
    anzahl_fahrten=("ausleihe_id", "count"),
    summe_km=...,   # TODO: distanz_km summieren
).reset_index()

nutzung.head()

## Schritt 4 — Das Label bilden: hat dieses Rad schon eine Meldung?

**TODO:** Bilden Sie aus `schadensmeldung` eine Menge (`set(...)`) der `fahrrad_id`-Werte,
die mindestens eine Meldung haben. Legen Sie dann in `fahrrad` eine neue Spalte
`hat_meldung` an: `1`, wenn die `fahrrad_id` in dieser Menge vorkommt, sonst `0`
(Tipp: `.isin(...)` gefolgt von `.astype(int)`).

In [ ]:
raeder_mit_meldung = ...  # TODO: set der fahrrad_id aus schadensmeldung
fahrrad["hat_meldung"] = ...  # TODO: fahrrad_id.isin(...).astype(int)

anteil = fahrrad["hat_meldung"].mean()
print(f"Anteil Räder mit mindestens einer Meldung: {anteil:.1%}")

## Schritt 5 — Alles zusammenführen und weitere Merkmale ableiten

**TODO:**
1. Führen Sie `fahrrad` und `nutzung` über `fahrrad_id` zusammen (`pd.merge`,
   `how="left"`). Räder ohne jede Fahrt hätten sonst fehlende Werte bei
   `anzahl_fahrten`/`summe_km` — füllen Sie diese mit `.fillna(0)` auf.
2. Leiten Sie ein Merkmal `tage_im_bestand` ab: wie viele Tage liegt die Anschaffung
   zurück? Nehmen Sie als Stichtag den letzten Tag im Datensatz
   (`ausleihe["startzeit"].max()`).

In [ ]:
daten = pd.merge(fahrrad, nutzung, on="fahrrad_id", how="left")
daten[["anzahl_fahrten", "summe_km"]] = daten[["anzahl_fahrten", "summe_km"]].fillna(0)

stichtag = ausleihe["startzeit"].max()
daten["tage_im_bestand"] = ...  # TODO: (stichtag - angeschafft_am) in Tagen, als Zahl

daten[["fahrrad_id", "typ_code", "anzahl_fahrten", "summe_km", "tage_im_bestand", "hat_meldung"]].head()

## Schritt 6 — Kategoriale Merkmale codieren und Trainings-/Testsplit

Genau wie in Notebook 1: `typ_code` (CITY/EBIKE/CARGO) ist kategorial und muss codiert
werden, bevor ein Modell damit rechnen kann.

**TODO:**
1. One-Hot-Encoding auf `typ_code` (mit `.astype(int)`, siehe Notebook 1).
2. Merkmalsmatrix `X` (Fahrtenzahl, Kilometer, Bestandsalter, Typ-Dummies) und Zielspalte
   `y` (`hat_meldung`) bilden.
3. `train_test_split` mit `test_size=0.25`, `random_state=42` — diesmal zusätzlich mit
   `stratify=y`. **Warum stratifizieren?** Weil die beiden Klassen unterschiedlich groß
   sind (nicht 50/50). Ohne `stratify` könnte der Zufall dazu führen, dass die
   Testdaten viel mehr oder viel weniger Meldungen enthalten als die Trainingsdaten —
   `stratify=y` sorgt dafür, dass beide Anteile gleich bleiben.

In [ ]:
typ_dummies = pd.get_dummies(daten["typ_code"], prefix="typ").astype(int)
daten_modell = pd.concat([daten, typ_dummies], axis=1)

merkmalsspalten = ["anzahl_fahrten", "summe_km", "tage_im_bestand"] + list(typ_dummies.columns)
X = daten_modell[merkmalsspalten]
y = daten_modell["hat_meldung"]

X_train, X_test, y_train, y_test = ...  # TODO: train_test_split mit test_size=0.25, random_state=42, stratify=y

print("Training:", X_train.shape, "| Test:", X_test.shape)
print("Anteil Meldungen Training:", y_train.mean().round(2), "| Test:", y_test.mean().round(2))

## Schritt 7 — Den Entscheidungsbaum trainieren

Ein Entscheidungsbaum stellt nacheinander einfache Ja/Nein-Fragen an die Merkmale (etwa
"Summe km > 900?") und teilt die Räder so schrittweise in immer reinere Gruppen. Der
große Vorteil für den Einstieg: **man kann ihn sich ansehen** und nachvollziehen, warum
er eine Entscheidung trifft — anders als bei vielen anderen Verfahren.

Wir begrenzen die Tiefe (`max_depth=3`), damit der Baum überschaubar bleibt und nicht
jede zufällige Eigenheit der Trainingsdaten auswendig lernt (dieselbe
Overfitting-Gefahr wie in Notebook 1, hier aber direkt über einen Parameter gesteuert).

**TODO:** Erzeugen Sie `DecisionTreeClassifier(max_depth=3, random_state=42)` und
trainieren Sie ihn.

In [ ]:
baum = ...  # TODO: DecisionTreeClassifier(max_depth=3, random_state=42)
...  # TODO: baum.fit(...)

plt.figure(figsize=(16, 8))
plot_tree(baum, feature_names=merkmalsspalten, class_names=["keine Meldung", "Meldung"],
          filled=True, fontsize=9)
plt.show()

**Den Baum lesen:** Jeder Kasten zeigt die Bedingung, nach der aufgeteilt wird, sowie
`gini` (ein Maß für die "Unreinheit" der Gruppe — 0 heißt: alle Räder in diesem Kasten
gehören zur selben Klasse), `samples` (wie viele Trainingsräder hier landen) und `value`
(die Aufteilung auf die beiden Klassen). Wenig überraschend sollte die Wurzel des Baums
etwas mit den kumulierten Kilometern zu tun haben — das war ja genau das eingebaute
Verschleisssignal.

## Schritt 8 — Evaluation mit der Confusion-Matrix

Auf der Evaluation-Folie in Block 05 steht für Klassifikation: *"Confusion-Matrix,
Kosten von Fehlalarm gegen verpassten Alarm abwägen."* Eine einzelne Genauigkeits-Zahl
("90 % richtig!") verschleiert, **welche Art** von Fehler das Modell macht — und genau
die Art des Fehlers entscheidet, ob das Modell im Betrieb nützlich ist.

Es gibt zwei Fehlerarten:
- **Falsch positiv** (Fehlalarm): Modell sagt "Meldung wahrscheinlich", es kommt aber
  keine → kostet eine unnötige Inspektion.
- **Falsch negativ** (verpasster Alarm): Modell sagt "unwahrscheinlich", es kommt aber
  doch eine → kostet einen Ausfall auf der Straße.

**TODO:**
1. Sagen Sie mit `baum.predict(X_test)` die Klassen für die Testdaten vorher.
2. Bilden Sie die Confusion-Matrix mit `confusion_matrix(y_test, vorhersage)` und
   stellen Sie sie mit `ConfusionMatrixDisplay(...).plot()` dar.

In [ ]:
vorhersage = ...  # TODO: baum.predict(X_test)

cm = ...  # TODO: confusion_matrix(y_test, vorhersage)
ConfusionMatrixDisplay(cm, display_labels=["keine Meldung", "Meldung"]).plot(cmap="Blues")
plt.show()

print(classification_report(y_test, vorhersage, target_names=["keine Meldung", "Meldung"]))

**Frage zum Nachdenken:** Schauen Sie sich die Confusion-Matrix an. Welche Fehlerart
kommt bei Ihrem Baum häufiger vor — Fehlalarm oder verpasster Alarm? Wenn VeloCity
lieber ein paar unnötige Inspektionen in Kauf nimmt, als ein Rad mit Defekt auf der
Straße zu haben (Sicherheitsaspekt!): Würden Sie den Baum eher "vorsichtiger" oder
"nachlässiger" einstellen wollen — und über welchen Hebel könnte man das steuern
(Stichwort: `max_depth`, oder Gewichtung der Klassen über `class_weight="balanced"`)?

*Ihre Antwort hier …*

## Zusammenfassung

In diesem Notebook haben Sie:
- Merkmale auf Rad-Ebene durch Aggregation von Fahrtdaten gebildet (`.groupby()`),
- ein Klassifikationslabel aus einer separaten Tabelle abgeleitet, statt es künstlich
  vorzugeben,
- einen Entscheidungsbaum trainiert, begrenzt und visualisiert,
- mit `stratify` einen fairen Trainings-/Testsplit bei unausgeglichenen Klassen gebaut,
- eine Confusion-Matrix gelesen und zwei Fehlerarten gegeneinander abgewogen, statt nur
  eine einzelne Kennzahl zu betrachten.

**Weiter geht's mit Notebook 3 — Clustering:** Dort gruppieren wir Stationen nach ihrem
Nutzungsmuster, ganz ohne vorgegebene Kategorien.